# ワークフローの基本機能

このノートブックでは、フレキシブルワークフローの基本構成を学びます。

フレキシブルワークフローでは、次のように、役割の異なるノードを組み合わせたワークフローを構成します。

- タスクノード：ユーザーと会話しながら必要な情報を聞き出して、情報が揃ったら次のノードに進む
- エージェントノード：ユーザーとの会話は行わず、インストラクションで指示された処理を1回だけ行う
- 関数ノード：通常の関数で直前のノードの出力結果を受け取り、必要な処理を行う
- 分岐処理：特定のノードの処理結果に応じて、次に進むノードを決定する

## 事前準備

**[WBF-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[WBF-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[WBF-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[WBF-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

**[WBF-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk import Event, Workflow
from google.adk.agents.llm_agent import LlmAgent
from google.adk.workflow import DEFAULT_ROUTE

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[WBF-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

ワークフローの進捗にあわせて結果を表示する `async_output` オプションを追加しています。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message, async_output=False):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
                    if async_output:
                        display(Markdown(response))
        if not async_output:
            return '\n'.join(result)

## タスクノードとエージェントノードの定義

**[WBF-07]**

ユーザーの情報（名前と趣味）を収集するタスクノードを定義します。

In [6]:
class UserInformation(BaseModel):
    """ユーザーの情報"""
    name: str = Field(description='ユーザーの名前')
    hobby: str = Field(description='ユーザーの趣味')

instruction = '''
# タスク
1. ユーザーの名前と趣味を質問します。
2. 得られた情報を UserInformation にセットします。
3. UserInformation の情報が揃ったらタスクを終了します。

# 条件
- フレンドリーに会話してください。
- できるだけ名前と趣味をまとめて聞いてください。
- コンテキストから判断せずに、必ずユーザーに確認してから情報をセットすること。
'''

user_information_task = LlmAgent(
    name='user_information_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=UserInformation,
    description='ユーザーの情報を集めるタスク',
    instruction=instruction,
)

**[WBF-08]**

タスクノードの実行結果をメッセージとして出力する関数ノードを用意します。

In [7]:
async def output_user_information(node_input: UserInformation):
    message = f'''
```
=== ユーザー情報 ===
・ 名前: {node_input.name}
・ 趣味: {node_input.hobby}
```
'''
    return Event(message=message)

**[WBF-09]**

ユーザーへのあいさつのメッセージを出力するエージェントノードを定義します。

In [8]:
instruction = '''
ユーザー情報に基づいて、一文の簡単なあいさつを出力します。
'''

greeting_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-3.5-flash-lite',
    description='ユーザーにあいさつするエージェント',
    instruction=instruction,
    include_contents='default',
)

## ワークフローグラフの定義とワークフローの実行

**[WBF-10]**

ここまでに用意したノードを繋げたワークフローグラフを定義します。

In [9]:
greeting_workflow = Workflow(
    name='greeting_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            greeting_agent,
        ),
    ],
)

greeting_workflow_app = AdkApp(
    agent=greeting_workflow,
    app_name='greeting_workflow_app',
)

**[WBF-11]**

最初のメッセージを入力します。

In [23]:
chat_client = ChatClient(greeting_workflow_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

こんにちは！お話しできて嬉しいです。
差し支えなければ、あなたの「お名前」と「ご趣味」を教えていただけますか？

**[WBF-12]**

名前と趣味を聞かれていますが、あえて名前だけを返答します。

In [24]:
query = '''
山田太郎です。
'''
await chat_client.async_stream_query(query, async_output=True)

山田太郎さん、こんにちは！
お名前を教えていただきありがとうございます。

もう一つ、差し支えなければ山田さんの「ご趣味」についても教えていただけますか？

**[WBF-13]**

趣味も教えるように促されるので、追加で趣味を返答します。

集められた情報がメッセージとして表示されて、さらにこれに基づいて、あいさつの文章が生成されます。

In [25]:
query = '''
趣味は野球です。キャッチャーです。
'''
await chat_client.async_stream_query(query, async_output=True)


```
=== ユーザー情報 ===
・ 名前: 山田太郎
・ 趣味: 野球（キャッチャー）
```


山田太郎さん、こんにちは！キャッチャーとしての熱い野球トーク、いつでもお待ちしております。

**[WBF-14]**

セッション情報に記録された会話履歴を確認します。

In [34]:
session = await chat_client.adk_app.async_get_session(
    user_id = 'default_user',
    session_id = chat_client.session_id,
)

for i, event in enumerate(session.events):
    print(f'\n=== [Event {i+1}] ===')
    print(f'author: {event.author}')
    print(f'content: {event.content}')


=== [Event 1] ===
author: user
content: parts=[Part(
  text="""
こんにちは。
"""
)] role='user'

=== [Event 2] ===
author: user_information_task
content: parts=[Part(
  text="""こんにちは！お話しできて嬉しいです。
差し支えなければ、あなたの「お名前」と「ご趣味」を教えていただけますか？""",
  thought_signature=b'\x01\x8f=k_\xc9\x06\x94\xd8\x84\xd5\xcb\x8a\xb9\xaf\xcb\xa2}\xf7\x85\x1a\xf1\xe6\xf7`\xc2\xee\xaeW\xd8\x84\xd0E&\xb4\x9b+yq\x1d\xb3\xd4\xc2 \xe3\x9c\xcbD\x88\x8a\x1e\xb2\xd9C\xc7\xb5v\xa8_\xf6\x1d\xfd\x16@9\x93@\xd7\x98\x95u\xba\xc37\xbc\xc8\x00\xc6"j\xb0\xa3\xd9'
)] role='model'

=== [Event 3] ===
author: user
content: parts=[Part(
  text="""
山田太郎です。
"""
)] role='user'

=== [Event 4] ===
author: user_information_task
content: parts=[Part(
  text="""山田太郎さん、こんにちは！
お名前を教えていただきありがとうございます。

もう一つ、差し支えなければ山田さんの「ご趣味」についても教えていただけますか？""",
  thought_signature=b'\x01\x8f=k_\xe2\x8d\xc0\xf1`\x81\x05T~~s\xc1\x91]\x9f<_\xfb\xf6\r\xb3y\x01\xc1E\x06V\xce\x86c\xd9\xdb\xd7\xad\x8bII\xc2\xf4\x8bG\x84OiY\xc1\xbd5B\xbeh\xba\x97U\xc3\xb9\xa3ewRl\x8b\x07u\xeb

## 分岐処理の実装

**[WBF-15]**

ワークフローを再実行するか確認するタスクノードと、その後処理をする関数ノードを定義します。

In [35]:
class HumanCheckResult(BaseModel):
    """ワークフロー再実行の判断結果"""
    restart: bool = Field(description='判断結果')

instruction = '''
# タスク
1. ワークフローを再実行するかユーザーに質問します。
2. 得られた情報を HumanCheckResult.restart にセットします。
  - 再実行する場合は True
  - 再実行しない場合は False
3. HumanCheckResult.restart をセットしたらタスクを終了します。

# 条件
- 余計な会話はしないで、「はい」か「いいえ」の判断を求めてください。
- コンテキストから判断せずに、必ずユーザーに確認してから情報をセットすること。
'''

human_check_task = LlmAgent(
    name='human_check_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=HumanCheckResult,
    description='ワークフロー再実行の判断を受け取るタスク',
    instruction=instruction,
)

def process_human_check_result(node_input: HumanCheckResult):
    if node_input.restart:
        return Event(
            message='=== ワークフローを再実行します ===',
            route='restart',
        )
    else:
        return Event(
            message='=== ワークフローを終了します ===',
            route='end',
        )


**[WBF-16]**

分岐処理を追加したワークフローを定義ます。

分岐処理は、関数ノードが出力した Event オブジェクトの `restart` オプションの値で次のノードを決定します。

In [36]:
async def end_node():
    return Event(message='=== ワークフローが完了しました ===')

greeting_workflow = Workflow(
    name='greeting_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            greeting_agent,
            human_check_task, process_human_check_result,
        ),
        (
            process_human_check_result,
            {
                'restart': user_information_task,
                DEFAULT_ROUTE: end_node,
            },
        ),
        (
            end_node,
        )
    ],
)

greeting_workflow_app = AdkApp(
    agent=greeting_workflow,
    app_name='greeting_workflow_app',
)

**[WBF-17]**

最初のメッセージを入力します。

In [43]:
chat_client = ChatClient(greeting_workflow_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

こんにちは！
はじめまして。これからいくつかお伺いしたいのですが、あなたの「お名前」と「趣味」を教えていただけますか？

**[WBF-18]**

ユーザーの情報を入力します。

In [44]:
query = '''
山田太郎です。趣味は野球をすること。
'''
await chat_client.async_stream_query(query, async_output=True)


```
=== ユーザー情報 ===
・ 名前: 山田太郎
・ 趣味: 野球をすること
```


山田太郎さん、こんにちは！野球の調子はいかがですか？今日も良い一日になりますように。

ワークフローを再実行しますか？（はい / いいえ）

**[WBF-19]**

再実行の確認に「はい」で答えます。

In [45]:
query = '''
はい。お願いします。
'''
await chat_client.async_stream_query(query, async_output=True)

=== ワークフローを再実行します ===

こんにちは！お話しできて嬉しいです。
差し支えなければ、あなたの「お名前」と「趣味」を教えていただけますか？

**[WBF-20]**

ユーザーの情報を入力します。

In [46]:
query = '''
山田サチ子。野球の応援が好き。
'''
await chat_client.async_stream_query(query, async_output=True)


```
=== ユーザー情報 ===
・ 名前: 山田サチ子
・ 趣味: 野球の応援
```


山田サチ子さん、こんにちは！野球観戦の応援、いつもお疲れ様です！

ワークフローを再実行しますか？（「はい」または「いいえ」でお答えください）

**[WBF-21]**

再実行の確認に「中止してください」と答えます。

「はい」「いいえ」以外の表現でも正しく理解されることがわかります。

In [47]:
query = '''
中止してください。
'''
await chat_client.async_stream_query(query, async_output=True)

=== ワークフローを終了します ===


=== ワークフローが完了しました ===
